# Validate Data with Pydantic

**Course author:** Jayneel Girnara

Compare plain Python objects with Pydantic models, then build field constraints and nested models for products, orders, and carts.

## Learning goals
- Describe the difference between an unvalidated object and a validated model.
- Declare fields with Python types and `Field` constraints.
- Validate nested objects and collections of models.
- Inspect how annotations affect parsing and validation.

## Before you start
Install the repository dependencies. This notebook's core examples run locally and do not need an LLM API key. Replace personal or real-world data with synthetic examples before publishing or reusing the notebook.

## How to work through it
Run cells in order so each class and example object exists before the next section uses it. Read the validation constraints before constructing each model, and observe the validation error when an input violates a rule.

## 1. Start with an unvalidated Python class

A regular Python class lets attributes hold values of any type unless your code checks them. Assigning the age as a string succeeds, so downstream code must handle unexpected shapes itself.

**Checkpoint:** What type does the `age` attribute actually hold after this assignment?

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
class User:
    name=""
    email=""
    age=0

In [ ]:
user=User()
user.name="Alex Example"
user.email="alex@example.com"
user.age="19"

print(user)

In [ ]:
user.age

## 2. Add Pydantic model validation

A Pydantic `BaseModel` declares fields with annotations. Constructing a model parses compatible inputs and raises a validation error when values cannot satisfy the declared types. This is structural/type validation, not a check that a name or age is factually true.

**Checkpoint:** What happens if a non-numeric string is supplied for an integer field?

In [25]:
from pydantic import BaseModel,Field
from typing import List,Annotated


In [ ]:
class User(BaseModel):
    name: str 
    age: int 




In [ ]:
user=User(name="Alex Example",age=19)
user

## 3. Constrain individual fields

`Field` adds constraints beyond the Python type, such as a maximum title length, minimum description length, or numeric range. The input must satisfy both the type and the constraints.

**Checkpoint:** Which constraint rejects a price outside the permitted range?

In [ ]:
class Product(BaseModel):
    title:str = Field(max_length=100)
    description:str = Field(min_length=20)
    price:int = Field(gt=0,le=100)


In [ ]:
product=Product(title="SamsungS23",description="This is an amazing device from samsung ... ",price=20)
product2=Product(title="SamsungS25",description="This is an amazing device from samsung ... ",price=40)

## 4. Validate nested models

A model can use another Pydantic model as a field. Here an `Order` contains a `Product`, so Pydantic validates the nested product as part of validating the order.

**Checkpoint:** Why is a nested model better than storing the product as an arbitrary dictionary?

In [14]:
class Order(BaseModel):
    id:str = Field(min_length=6,max_length=12)
    product:Product

In [16]:
order=Order(id="i23sgnj",product=product)
order

Order(id='i23sgnj', product=Product(tittle='SamsungS23', des='This is an amazing device from samsung ... ', price=20))

In [18]:
order.product.price

20

## 5. Validate a collection of models

A list field can contain many `Product` objects. `Annotated` attaches validation metadata to a type; use `Field(min_length=1)` to require at least one item. The collection is validated item by item as well as against its own constraints.

**Checkpoint:** Which constraint ensures that a cart cannot have an empty item list?

In [20]:
class Carts(BaseModel):
    id:str = Field(min_length=6,max_length=12)
    items:List[Product]
    comment:str = Field(default="No comment")


In [24]:
cart=Carts(id="1234fdav",items=[product,product2])
cart

Carts(id='1234fdav', items=[Product(tittle='SamsungS23', des='This is an amazing device from samsung ... ', price=20), Product(tittle='SamsungS25', des='This is an amazing device from samsung ... ', price=40)], comment='No comment')

In [ ]:
ProductList = Annotated[List[Product], Field(min_length=1)]

In [ ]:
class Carts(BaseModel):
    id:str = Field(min_length=6,max_length=12)
    items:ProductList
    comment:str = Field(default="No comment")
